<a href="https://colab.research.google.com/github/kavyasudha12/Gen-AI-Training-Task/blob/main/SIMPLE_MEMORY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install dependencies
!pip install -q openai requests beautifulsoup4

# -------------------------
# 0. SET ENV VARIABLE
# -------------------------
import os

os.environ["OPENAI_API_KEY"] = input("Paste your API key here: ").strip()

# -------------------------
# Imports
# -------------------------
from openai import OpenAI
import requests
from bs4 import BeautifulSoup
import json

# Initialize client (no key needed here now)
client = OpenAI()


# -------------------------
# 1. SIMPLE MEMORY
# -------------------------
memory = []

def add_to_memory(role, content):
    memory.append({"role": role, "content": content})

def get_recent_memory(n=5):
    return memory[-n:]


# -------------------------
# 2. TOOL: FETCH WEBPAGE
# -------------------------
def fetch_webpage(url):
    try:
        response = requests.get(url, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")

        for tag in soup(["script", "style"]):
            tag.extract()

        text = soup.get_text(separator="\n")
        return text[:3000]

    except Exception as e:
        return f"Error fetching page: {str(e)}"


# -------------------------
# 3. TOOL DEFINITION
# -------------------------
tools = [
    {
        "type": "function",
        "name": "fetch_webpage",
        "description": "Fetch content from a webpage URL",
        "parameters": {
            "type": "object",
            "properties": {
                "url": {
                    "type": "string",
                    "description": "URL of the webpage"
                }
            },
            "required": ["url"]
        }
    }
]


# -------------------------
# 4. AGENT LOOP
# -------------------------
def run_agent(user_input):
    add_to_memory("user", user_input)

    context = "\n".join(
        [f"{m['role']}: {m['content']}" for m in get_recent_memory()]
    )

    response = client.responses.create(
        model="gpt-5",
        input=f"""
You are an assistant with memory and tools.

Conversation:
{context}

User: {user_input}
""",
        tools=tools
    )

    for item in response.output:
        if item.type == "function_call":

            tool_name = item.name
            args = json.loads(item.arguments)

            print(f"🔧 Tool called: {tool_name}")

            if tool_name == "fetch_webpage":
                result = fetch_webpage(args["url"])

            response2 = client.responses.create(
                model="gpt-5",
                previous_response_id=response.id,
                input=[
                    {
                        "type": "function_call_output",
                        "call_id": item.call_id,
                        "output": result
                    }
                ]
            )

            final_answer = response2.output_text
            add_to_memory("assistant", final_answer)
            return final_answer

    final_answer = response.output_text
    add_to_memory("assistant", final_answer)
    return final_answer


# -------------------------
# 5. DEMO
# -------------------------
print(run_agent("Summarize this page: https://en.wikipedia.org/wiki/Artificial_intelligence"))